Goal: able to find dependency between the property and the associated numerical value

In [8]:
import torch
import transformers
from transformers import BertTokenizerFast, BertForTokenClassification
from TorchCRF import CRF
from torch.utils.data import DataLoader, Dataset
import numpy as np

In [9]:
import datasets
import import_ipynb
import spacy
from Extract_Relationship import *

In [10]:
label_map = {
    "O": 0,
    "B-PROPERTY": 1, #主语
    "I-PROPERTY": 2,
    "B-YEAR": 3, #year
    "I-YEAR": 4, 
    "B-TIME": 5, #quarter, month, 
    "I-TIME": 6,
    "B-UNIT": 7, #currency, dollar sign, etc. optional, some value may not have a unit
    "I-UNIT": 8,
    "B-VALUE": 9, #number only
    "I-VALUE": 10, 
    "B-MULTIPLIER": 11,
    "I-MULTIPLIER": 12,
    "B-CHANGE": 13, #increase, decrease, etc. for dependency parsing, to be implemented
    "I-CHANGE": 14,
    "B-CHANGE_UNIT": 15, #%, basis point, etc
    "I-CHANGE_UNIT": 16,
    "B-COMPANY": 17, #if we want to compare different companies
    "I-COMPANY": 18
}

In [11]:
train_sentences = [
    "GMV in 2023 was $ 19.2 billion",
    "Revenue in 2021 was $ 4.38 billion",
    "Revenue in 2022 was $ 5.56 billion",
    "Revenue in 2023 was $ 7.68 billion",
    "Profit in 2021 was $ 24 million",
    "Profit in 2022 was $ 372 million",
    "Profit in 2023 was $ 1.10 billion",
    "Adjusted profit in 2021 was $ 769.6 million",
    "Adjusted profit in 2022 was $ 788.1 million",
    "Adjusted profit in 2023 was $ 1.46 billion"
]

train_labels = [
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"]
]

train_sentences.extend([
    "Tesla 's revenue in 2023 was $ 81.5 billion",
    "Amazon 's net profit in 2022 was $ 33.3 billion",
    "Apple 's operating income in 2024 was $ 34.2 billion",
    "Google 's advertising revenue in 2023 was $ 280 billion",
    "Microsoft 's cloud revenue in 2022 was $ 72 billion",
    "Facebook 's profit margin in 2023 was 25 %",
    "Netflix 's subscriber growth in 2023 was 12 %",
    "Goldman Sachs 's investment banking revenue in 2021 was $ 14.5 billion",
    "JP Morgan 's total assets in 2023 were $ 3.9 trillion",
    "Berkshire Hathaway 's total revenue in 2023 was $ 302 billion"
])

train_labels.extend([
    ["B-COMPANY", "O", "B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-UNIT"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-UNIT"],
    ["B-COMPANY", "I-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "I-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "I-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"]
])


Preprocess:
aligns tokenized words with assigned labels 

In [12]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

In [13]:
def align_labels(sentence, word_labels):
    words = sentence.split()
    tokens = []
    aligned_labels = []

    word_idx = 0  # of original label

    for word in words:
        sub_tokens = tokenizer.tokenize(word)
        tokens.extend(sub_tokens)

        first_label = word_labels[word_idx]

        # If it's the first word of an entity, keep it as "B-"
        # If it's an "I-" label originally, it should stay "I-"
        sub_labels = [first_label] + [
            "I-" + first_label[2:] if first_label.startswith("B-") else first_label
            for _ in range(len(sub_tokens) - 1)
        ]

        aligned_labels.extend(sub_labels)
        word_idx += 1

    return tokens, aligned_labels

In [14]:
test_sentence = "Net income for 2021 was 4.5 billion dollars"
test_labels = ["B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"]

tokens, adjusted_labels = align_labels(test_sentence, test_labels)

print(f"Original Sentence: {test_sentence}")
print(f"Tokenized Output: {tokens}")
print(f"Aligned Labels: {adjusted_labels}")

Original Sentence: Net income for 2021 was 4.5 billion dollars
Tokenized Output: ['net', 'income', 'for', '2021', 'was', '4', '.', '5', 'billion', 'dollars']
Aligned Labels: ['B-PROPERTY', 'I-PROPERTY', 'O', 'B-YEAR', 'O', 'B-VALUE', 'I-VALUE', 'I-VALUE', 'B-MULTIPLIER', 'B-UNIT']


In [41]:
# for i, sentence in enumerate(train_sentences):
#     tokens, adjusted_labels = align_labels(sentence, train_labels[i])
#     print(f"Sentence {i}: {sentence}")
#     print(f"Tokenized: {tokens}")
#     print(f"Aligned Labels: {adjusted_labels}")
#     print("-" * 40)

In [43]:
def tokenize_and_align_labels(sentences, labels, max_length=20):
    tokenized_inputs = {"input_ids": [], "attention_mask": [], "labels": []}

    for i, (sentence, word_labels) in enumerate(zip(sentences, labels)):
        tokens, aligned_labels = align_labels(sentence, word_labels)

        # [CLS] and [SEP], denoting the start and end of a sentence
        tokens = ["[CLS]"] + tokens + ["[SEP]"]
        aligned_labels = ["O"] + aligned_labels + ["O"] 

        # Convert tokens to input IDs
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        attention_mask = [1] * len(input_ids)

        # Convert aligned labels to numerical values using `label_map`
        label_ids = [label_map.get(lbl, 0) for lbl in aligned_labels]

        # Pad sequences to max_length
        padding_length = max_length - len(input_ids)
        if padding_length > 0:
            input_ids += [0] * padding_length  # Pad input IDs with 0 (BERT's padding)
            attention_mask += [0] * padding_length  # Pad attention mask with 0
            label_ids += [-100] * padding_length  # Pad labels with -100 to ignore them in loss

        tokenized_inputs["input_ids"].append(input_ids)
        tokenized_inputs["attention_mask"].append(attention_mask)
        tokenized_inputs["labels"].append(label_ids)

        # Debugging Output
        # print(f"\nSentence {i}: {sentence}")
        # print(f"Tokenized: {tokens}")
        # print(f"Aligned Labels: {aligned_labels}")
        # print(f"Input IDs: {input_ids}")
        # print(f"Attention Mask: {attention_mask}")
        # print(f"Label IDs: {label_ids}")
        # print("-" * 60)

    return tokenized_inputs


In [44]:
train_encodings = tokenize_and_align_labels(train_sentences, train_labels)

In [45]:
class NERDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()} 

In [46]:
train_dataset = NERDataset(train_encodings)
train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)

In [48]:
class BertCRF(torch.nn.Module):
    def __init__(self, num_labels):
        super(BertCRF, self).__init__()
        self.bert = BertForTokenClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, return_dict=False)
        emissions = outputs[0]

        if labels is not None:
            labels = labels.clone()  # Avoid modifying original tensor
            labels[labels == -100] = 0  # Replace -100 with 'O'

            # Compute CRF loss
            loss = -self.crf(emissions, labels, mask=attention_mask.bool(), reduction="mean")
            return loss
        else:
            return self.crf.decode(emissions, mask=attention_mask.bool())


In [49]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertCRF(num_labels=len(label_map)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
def train_model(num_epochs=3):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for batch in train_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            loss = model(input_ids, attention_mask, labels)
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_dataloader):.4f}")

In [23]:
train_model(num_epochs=10)

Epoch 1 Loss: 34.3949
Epoch 2 Loss: 20.1145
Epoch 3 Loss: 12.7405
Epoch 4 Loss: 7.5919
Epoch 5 Loss: 5.1533
Epoch 6 Loss: 3.1709
Epoch 7 Loss: 2.3732
Epoch 8 Loss: 1.6213
Epoch 9 Loss: 1.1357
Epoch 10 Loss: 0.7973


In [24]:
def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt")
    inputs.pop("token_type_ids", None)

    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        predictions = model(**inputs)

    predicted_labels = [list(label_map.keys())[pred] for pred in predictions[0]]
    tokens = tokenizer.tokenize(tokenizer.decode(inputs["input_ids"][0]))

    return list(zip(tokens, predicted_labels))

In [26]:
test_text = "net income increased to 20 million in 2024"
print(predict(test_text))

test_text2 = "In 2023, Apple’s revenue grew to 200 billion dollars, while Microsoft reported revenue of 180 billion dollars. Google’s net profit in 2022 was 50 billion dollars. Amazon’s operating expenses in 2021 totaled 150 billion dollars."
print(predict(test_text2))

[('[CLS]', 'O'), ('net', 'B-PROPERTY'), ('income', 'I-PROPERTY'), ('increased', 'O'), ('to', 'O'), ('20', 'B-VALUE'), ('million', 'B-MULTIPLIER'), ('in', 'O'), ('202', 'B-YEAR'), ('##4', 'I-YEAR'), ('[SEP]', 'O')]
[('[CLS]', 'O'), ('in', 'O'), ('202', 'B-YEAR'), ('##3', 'I-YEAR'), (',', 'O'), ('apple', 'B-COMPANY'), ('’', 'O'), ('s', 'O'), ('revenue', 'I-PROPERTY'), ('grew', 'O'), ('to', 'O'), ('200', 'B-VALUE'), ('billion', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), (',', 'O'), ('while', 'O'), ('microsoft', 'B-COMPANY'), ('reported', 'O'), ('revenue', 'I-PROPERTY'), ('of', 'O'), ('180', 'B-VALUE'), ('billion', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), ('.', 'O'), ('google', 'B-COMPANY'), ('’', 'O'), ('s', 'O'), ('net', 'B-PROPERTY'), ('profit', 'I-PROPERTY'), ('in', 'O'), ('202', 'B-YEAR'), ('##2', 'I-YEAR'), ('was', 'O'), ('50', 'B-VALUE'), ('billion', 'B-MULTIPLIER'), ('dollars', 'B-UNIT'), ('.', 'O'), ('amazon', 'B-COMPANY'), ('’', 'O'), ('s', 'O'), ('operating', 'B-PROPERTY'), ('expens

In [72]:
from collections import defaultdict

def check_consistency(sentences):
    """Checks if the same property is consistently linked to the same value in each year."""
    property_values_by_year = defaultdict(lambda: defaultdict(set))  # {year: {property: set(values)}}

    for sentence in sentences:
        predictions = predict(sentence)  # Run model prediction
        current_property = None
        current_year = None
        current_value = []

        triplets = set()  # Store complete (property, year, value) triplets

        for token, label in predictions:
            # Track the year
            if label.startswith("B-YEAR"):
                current_year = token
            elif label.startswith("I-YEAR") and current_year:
                current_year += token.replace("##", "")  # Merge subword years like "202" + "##3" → "2023"

            # Track the property
            if label.startswith("B-PROPERTY"):
                if current_property and current_value:  # Store previous triplet if it exists
                    triplets.add((current_year, current_property, " ".join(current_value)))
                current_property = token
                current_value = []  # Reset value tracking
            elif label.startswith("I-PROPERTY") and current_property:
                current_property += " " + token.replace("##", "")  # Merge subword properties

            # Track the value
            if label.startswith("B-VALUE") or label.startswith("I-VALUE") or label.startswith("B-MULTIPLIER"):
                current_value.append(token.replace("##", ""))  # Ensure subword merging

        # Store final triplet after finishing sentence
        if current_property and current_value:
            triplets.add((current_year, current_property, " ".join(current_value)))

        # Store triplets in the dictionary
        for year, prop, val in triplets:
            if year and prop and val:  # Ensure all parts exist
                property_values_by_year[year][prop].add(val)

        # # Debugging Output
        # print(f"\n🔹 Sentence: {sentence}")
        # print(f"🔸 Extracted Triplets: {dict(property_values_by_year)}")

    # 🔍 Identify inconsistencies
    print(f"Triplets: {dict(property_values_by_year)}")
    inconsistencies = {}
    for year, properties in property_values_by_year.items():
        for prop, values in properties.items():
            if len(values) > 1:  # If the same property has multiple values in the same year
                inconsistencies.setdefault(year, {})[prop] = values

    if inconsistencies:
        print("no")
        for year, props in inconsistencies.items():
            for prop, values in props.items():
                print(f"Year {year}: {prop} → {values}")
    else:
        print("pass")

    return property_values_by_year, inconsistencies


In [73]:
test_sentences = [
    "Revenue in 2023 was $10M",
    "Revenue in 2023 reached $10M",
    "The operating cost in 2023 was 10 million dollars",
    "The operating cost in 2023 reached 10 million dollars",
    "Net profit in 2022 amounted to $5M",
    "Net profit in 2023 amounted to $7M",
]

property_values, inconsistencies = check_consistency(test_sentences)


Triplets: {'2023': defaultdict(<class 'set'>, {'revenue': {'10 m'}, 'operating cost': {'10 million'}, 'profit': {'7 m'}}), '2022': defaultdict(<class 'set'>, {'profit': {'5 m'}})}
pass


In [74]:
test_sentences = [
    "Net profit in 2023 was $7M",
    "Net profit in 2023 was $5M",
    "The operating cost in 2022 was 10 million dollars",
    "The operating cost in 2022 reached 7 million dollars",
]
property_values, inconsistencies = check_consistency(test_sentences)

Triplets: {'2023': defaultdict(<class 'set'>, {'profit': {'5 m', '7 m'}}), '2022': defaultdict(<class 'set'>, {'operating cost': {'10 million', '7 million'}})}
no
Year 2023: profit → {'5 m', '7 m'}
Year 2022: operating cost → {'10 million', '7 million'}
